# Supplementary Figs. 2 and 3 — Task performance across learning-rate parameters

Per-rule $(\alpha,\beta)$ heatmaps of task performance, one figure per network size.
For each rule (row) the first six columns show each task's own performance field; the last
column overlays all six tasks' **good regions** translucently so their $(\alpha,\beta)$
overlap is visible.

**Conventions**
- The nuisance learning rates $\eta$ (and $\tau$, for the four $\tau$-rules) are **marginalized by
  max**: every $(\alpha,\beta)$ cell shows the *best achievable* performance over $\eta,\tau$.
  The `PC` $\eta$-example figure at the end shows what this envelope is built from.
- Each task is displayed on a **single-hue, higher-is-better** scale (white = worst, color = best),
  using the same identity color in its heatmap and in the overlay.
- Overlay = **absolute per-task threshold** on the higher-is-better metric (a rule that cannot do a
  task shows nothing there), drawn as a faint fill plus a crisp cell-aligned outline.
- Metric definitions follow Methods (*Task scores*). Reward-seeking uses the aggregated `det` tensor (the operating-point medians of Fig. 1g are scalars and can't fill a heatmap).
- **Matching pennies** is the *adaptive-unpredictability gap* from the settled-behaviour sweep
  (`matchingpennies_steady_*`, 7 alternating phases): median entropy against the competitive n-gram
  opponent minus median entropy against the action-independent 50/50 schedule (both 3-bit block entropy),
  the same quantity Fig 3j selects on. The older 3-phase sweep is in `data/sweeps/old_mp/`.
  This replaces raw entropy, which saturates near the 3-bit ceiling almost everywhere; the gap isolates
  whether a rule raises its unpredictability specifically when the opponent is exploiting it.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from matplotlib.patches import Patch
from matplotlib.collections import LineCollection
import matplotlib.ticker as mticker

def find_dir(candidates, what):
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(f"{what} not found. Tried: {candidates}")

# Sweep summaries live in data/sweeps/ and data/det/ (resolved relative to the working directory).
DATA_ROOT = find_dir(["../../../data", "../../data", "../data", "data", "./data"], "data/")
SWEEP_DIR = find_dir([f"{DATA_ROOT}/sweeps"], "data/sweeps/")
DET_DIR   = find_dir([f"{DATA_ROOT}/det"], "data/det/")
print("sweeps:", os.path.abspath(SWEEP_DIR))
print("det   :", os.path.abspath(DET_DIR))

DIST_FITS_DIR = f"{SWEEP_DIR}/dist_fits"
HAVE_DIST_FITS = os.path.isdir(DIST_FITS_DIR)
if not HAVE_DIST_FITS:
    print(f"NOTE: {DIST_FITS_DIR} not found -- 'heavytail' column will be blank.")

TAU_RULES = {'COV', 'BCM', 'LC', 'HZ'}

RULES = ['PC', 'HR', 'OJA', 'COV', 'BCM', 'LC', 'HZ']
LEVY_OPT = 2.0

# Sweep grids -- must match ssb/scripts/run_sweep.py and ssb.analysis.cell_select.
ETAS   = np.logspace(-6, -1, 6)
TAUS   = np.logspace(-3, -1, 4)
ALPHAS = np.logspace(-5, 0, 15)
BETAS  = np.logspace(-5, 0, 15)
EXT    = [-5, 0, -5, 0]            # imshow extent in log10(beta), log10(alpha)
EDGES  = np.linspace(-5, 0, 16)   # 16 cell edges for 15 cells

TASKS = ['selectivity', 'slowfeature', 'rseeking', 'heavytail', 'matchingpennies', 'probmatch']

# identity color (Okabe-Ito, colorblind-safe) + display scale (higher-is-better P units)
# + absolute good-region threshold used for the overlay mask.
CFG = {
 'selectivity':    dict(label='Feature\nselectivity', color='#CC79A7', vmin=0.0,  vmax=0.9, thr=0.60, cbar=r'selectivity $\chi$'),
 'slowfeature':    dict(label='Slow\nfeature',         color='#D55E00', vmin=0.0,  vmax=0.8, thr=0.40, cbar=r'min $\chi$ (maximin)'),
 'rseeking':       dict(label='Reward\nseeking',       color='#009E73', vmin=0.4,  vmax=1.0, thr=0.70, cbar=r'$P$(reward)'),
 'heavytail':      dict(label='Heavy\ntail',           color='#56B4E9', vmin=-0.6, vmax=0.0, thr=-0.30, cbar=r'$-|$exp$-2|$ (0 best)'),
 'matchingpennies':dict(label='Matching\npennies',     color='#E69F00', vmin=0.0,  vmax=2.0, thr=1.00, cbar='entropy gap (bits)'),
 'probmatch':      dict(label='Prob.\nmatching',       color='#0072B2', vmin=-1.0, vmax=0.0, thr=-0.50, cbar=r'$-|$match index$|$ (0 best)'),
}
FILL_ALPHA = 0.15   # faint translucent overlay fill
OUT_LW     = 1.05   # crisp identity-color outline

def cmap_for(color):
    return LinearSegmentedColormap.from_list('', ['#ffffff', color])

In [ ]:
def _to_etau(a):
    """Insert a length-1 tau axis for no-tau rules so everything is (eta, tau, alpha, beta)."""
    return a[:, None] if a.ndim == 3 else a

def load_task(task, rule, size):
    """Return (P, D): P = higher-is-better performance, D = display field.
    Both shape (n_eta, n_tau, 15, 15). (Same metric definitions as FGrid.ipynb;
    here D == P because every task is shown on its higher-is-better scale.)"""
    if task == 'selectivity':
        a = _to_etau(np.load(f'{SWEEP_DIR}/selectivity_{rule}_{size}.npy')); return a, a
    if task == 'slowfeature':                                  # maximin over features
        a = _to_etau(np.min(np.load(f'{SWEEP_DIR}/slowfeature_{rule}_{size}.npy'), axis=-1)); return a, a
    if task == 'rseeking':                                     # deterministic reward, P(reward)
        a = _to_etau(np.load(f'{DET_DIR}/det_{rule}_{size}.npy')); return a, a
    if task == 'heavytail':                                    # Levy exponent, best at -2
        n_tau = len(TAUS) if rule in TAU_RULES else 1
        if not HAVE_DIST_FITS:
            nan = np.full((len(ETAS), n_tau, 15, 15), np.nan)
            return nan, nan
        exp = np.load(f'{DIST_FITS_DIR}/{rule}_{size}_fits.npz')['exponent']
        P = -np.abs(exp - LEVY_OPT); return P, P               # 0 best, more negative worse
    if task == 'matchingpennies':                              # costly stochasticity (bits)
        # Settled-behaviour sweep (7 phases: computer, then 3 x [coin flip, computer]; 20k steps
        # each; scored over the 2nd half of each phase). Metric = median 3-bit entropy against the
        # computer (phases 2, 4, 6; 0-based) minus median entropy under the coin flip (phases 3, 5),
        # medians over seeds and phases -- the quantity Fig 3j's cells are selected on
        # (main_figs/code/f4_probmatching_matchingpennies/select_mp_cells.py).
        f = f'{SWEEP_DIR}/matchingpennies_steady_{rule}_{size}.npz'
        if not os.path.exists(f):                              # big-network sweep not finished yet
            n_tau = len(TAUS) if rule in TAU_RULES else 1
            nan = np.full((len(ETAS), n_tau, 15, 15), np.nan); return nan, nan
        S = np.load(f)['scores']                               # (eta, tau, alpha, beta, phase, metric, seed)
        med = lambda ps: np.nanmedian(S[..., ps, 0, :].reshape(S.shape[:4] + (-1,)), -1)
        d = med([2, 4, 6]) - med([3, 5]); return d, d
    if task == 'probmatch':                                    # matching index, 0 best
        a = _to_etau(np.load(f'{SWEEP_DIR}/bandit_bait_{rule}_{size}.npy')); return -np.abs(a), -np.abs(a)
    raise ValueError(task)

def max_over_etau(P):
    """Max over eta & tau (axes 0,1) -> (15,15). P is higher-is-better."""
    Pf = P.reshape(-1, 15, 15)
    return np.nanmax(np.where(np.isnan(Pf), -np.inf, Pf), axis=0)

In [ ]:
def style(ax, box='#C7CBD1', lw=0.4):
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor(box); sp.set_linewidth(lw)

def mask_edges(mask):
    """Line segments tracing the boundary of `mask` on the imshow cell grid, so the
    outline lands exactly on the fill's blocky edges (no cell-center contour drift)."""
    n = mask.shape[0]; segs = []
    for i in range(n):          # alpha row
        for j in range(n):      # beta col
            if not mask[i, j]:
                continue
            x0, x1 = EDGES[j], EDGES[j+1]; y0, y1 = EDGES[i], EDGES[i+1]
            if j == 0   or not mask[i, j-1]: segs.append([(x0, y0), (x0, y1)])
            if j == n-1 or not mask[i, j+1]: segs.append([(x1, y0), (x1, y1)])
            if i == 0   or not mask[i-1, j]: segs.append([(x0, y0), (x1, y0)])
            if i == n-1 or not mask[i+1, j]: segs.append([(x0, y1), (x1, y1)])
    return segs

def draw_overlay(ax, P_fields):
    """Light translucent fill + cell-aligned crisp outline per task."""
    ax.imshow(np.ones((15, 15, 3)), origin='lower', aspect='auto', extent=EXT)
    for task in TASKS:
        cf = CFG[task]; mask = P_fields[task] >= cf['thr']
        if not mask.any():
            continue
        rgba = np.zeros((15, 15, 4)); rgba[..., :3] = to_rgb(cf['color'])
        rgba[..., 3] = np.where(mask, FILL_ALPHA, 0.0)
        ax.imshow(rgba, origin='lower', aspect='auto', extent=EXT)
        segs = mask_edges(mask)
        if segs:
            ax.add_collection(LineCollection(segs, colors=cf['color'], linewidths=OUT_LW))
    ax.set_xlim(-5, 0); ax.set_ylim(-5, 0)

In [ ]:
def draw_grid(size, save_stem=None, dpi=200):
    """S2 main figure for one network size: rules (rows) x [6 tasks + overlay] (cols)."""
    nR, nC = len(RULES), len(TASKS) + 1
    fig, axes = plt.subplots(nR, nC, figsize=(nC*1.12, nR*1.28))
    ims = {}   # one representative mappable per task column (for its colorbar)
    for r, rule in enumerate(RULES):
        P_fields = {}
        for c, task in enumerate(TASKS):
            P, _ = load_task(task, rule, size)
            Pm = max_over_etau(P); P_fields[task] = Pm
            cf = CFG[task]; ax = axes[r, c]
            im = ax.imshow(Pm, cmap=cmap_for(cf['color']), origin='lower', aspect='auto',
                           extent=EXT, vmin=cf['vmin'], vmax=cf['vmax'])
            if r == 0: ims[task] = im
            style(ax)
            if r == 0: ax.set_title(cf['label'], fontsize=7, pad=3)
            if c == 0:
                ax.set_ylabel(rule, fontsize=9, fontweight='bold', rotation=0,
                              ha='right', va='center', labelpad=8)
        ax = axes[r, nC-1]
        draw_overlay(ax, P_fields)
        style(ax, box='#888', lw=0.6)
        if r == 0: ax.set_title('All tasks\n(overlay)', fontsize=7, pad=3)

    # axis-orientation hint on bottom-left panel (keeps the HZ rule label intact)
    a0 = axes[-1, 0]
    a0.set_xticks([-5, 0]); a0.set_yticks([-5, 0])
    a0.tick_params(labelsize=5, width=0.3, length=2, pad=1)
    a0.set_xlabel(r'$\log_{10}\beta$', fontsize=6, labelpad=1)
    a0.text(-0.78, 0.5, r'$\log_{10}\alpha$', transform=a0.transAxes, fontsize=6,
            rotation=90, ha='center', va='center')

    fig.suptitle(f'S2 - {size} networks: task performance vs learning rates '
                 r'($\alpha,\beta$; max over $\eta,\tau$)', fontsize=9, y=1.0)
    fig.tight_layout(rect=[0, 0.13, 1, 0.98])

    # one labeled colorbar under each task column (each task has its own scale/units)
    fig.canvas.draw()
    for c, task in enumerate(TASKS):
        pos = axes[-1, c].get_position()
        cax = fig.add_axes([pos.x0, 0.075, pos.width, 0.014])
        cb = fig.colorbar(ims[task], cax=cax, orientation='horizontal')
        cb.set_label(CFG[task]['cbar'], fontsize=5.5, labelpad=1.5)
        cb.ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
        cb.ax.tick_params(labelsize=4.5, width=0.3, length=1.8, pad=1)
        cb.outline.set_linewidth(0.3)

    # overlay key (outline colors) below the colorbars
    handles = [Patch(facecolor=CFG[t]['color'], alpha=0.55,
                     label=CFG[t]['label'].replace('\n', ' ')) for t in TASKS]
    fig.legend(handles=handles, fontsize=6.5, frameon=False, loc='lower center',
               ncol=6, bbox_to_anchor=(0.5, 0.005), title='overlay key',
               title_fontsize=6)
    if save_stem:
        for ext in ('svg', 'png'):
            fig.savefig(f'{save_stem}.{ext}', dpi=dpi, bbox_inches='tight')
        print('saved', save_stem + '.svg /.png')
    return fig

In [ ]:
def draw_pc_eta(size, save_stem=None, tasks=('rseeking', 'selectivity'), dpi=200):
    """Example (PC only): how the (alpha,beta) map moves across eta, which the main figure
    marginalizes. Columns = eta sweep (tau collapsed by max) + the max-over-(eta,tau) envelope."""
    nR, nC = len(tasks), len(ETAS) + 1
    fig, axes = plt.subplots(nR, nC, figsize=(nC*1.05, nR*1.15))
    if nR == 1: axes = axes[None, :]
    ims = {}
    for r, task in enumerate(tasks):
        P, _ = load_task(task, 'PC', size); cf = CFG[task]; cm = cmap_for(cf['color'])
        Pt = np.nanmax(np.where(np.isnan(P), -np.inf, P), axis=1)   # collapse tau -> (eta,15,15)
        for e in range(len(ETAS)):
            ax = axes[r, e]
            ax.imshow(Pt[e], cmap=cm, origin='lower', aspect='auto', extent=EXT,
                      vmin=cf['vmin'], vmax=cf['vmax'])
            style(ax)
            if r == 0: ax.set_title(rf'$\eta$=1e{int(round(np.log10(ETAS[e])))}', fontsize=6.5, pad=3)
            if e == 0:
                ax.set_ylabel(cf['label'], fontsize=8, rotation=0, ha='right', va='center', labelpad=8)
        ax = axes[r, nC-1]
        ims[task] = ax.imshow(max_over_etau(P), cmap=cm, origin='lower', aspect='auto', extent=EXT,
                              vmin=cf['vmin'], vmax=cf['vmax'])
        style(ax, box='#888', lw=0.6)
        if r == 0: ax.set_title('max over\n$\\eta,\\tau$', fontsize=6.5, pad=3)
    a0 = axes[-1, 0]; a0.set_xticks([-5, 0]); a0.set_yticks([-5, 0])
    a0.tick_params(labelsize=5, width=0.3, length=2, pad=1)
    a0.set_xlabel(r'$\log_{10}\beta$', fontsize=6); a0.set_ylabel(r'$\log_{10}\alpha$', fontsize=6)
    fig.suptitle(f'S2 example - PC, {size}: (alpha,beta) map across eta '
                 '(marginalized in main panel)', fontsize=8.5, y=1.02)
    fig.tight_layout(rect=[0, 0, 0.9, 1])

    # one labeled colorbar per row (task), to the right of the max-envelope panel
    fig.canvas.draw()
    for r, task in enumerate(tasks):
        pos = axes[r, -1].get_position()
        cax = fig.add_axes([0.915, pos.y0, 0.012, pos.height])
        cb = fig.colorbar(ims[task], cax=cax, orientation='vertical')
        cb.set_label(CFG[task]['cbar'], fontsize=5.5, labelpad=2)
        cb.ax.yaxis.set_major_locator(mticker.MaxNLocator(3))
        cb.ax.tick_params(labelsize=4.5, width=0.3, length=1.8, pad=1)
        cb.outline.set_linewidth(0.3)
    if save_stem:
        for ext in ('svg', 'png'):
            fig.savefig(f'{save_stem}.{ext}', dpi=dpi, bbox_inches='tight')
        print('saved', save_stem + '.svg /.png')
    return fig

## Render the figures

In [ ]:
fig = draw_grid('small', save_stem='S2_small'); plt.show()
fig = draw_grid('big',   save_stem='S2_big');   plt.show()
fig = draw_pc_eta('small', save_stem='S2_pc_eta_small'); plt.show()